In [13]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from peft import LoraConfig, TaskType, get_peft_model
from transformers import T5EncoderModel, T5ForConditionalGeneration
from transformers.modeling_outputs import BaseModelOutput


In [22]:


@dataclass
class T5EncoderOutput:
    last_hidden_state: torch.Tensor


@dataclass
class T5DecoderOutput:
    last_hidden_state: torch.Tensor
    all_hidden_states: Optional[Tuple[torch.Tensor, ...]] = None


@dataclass
class FactorVAEOutput:
    z: torch.Tensor
    mu: torch.Tensor
    logvar: torch.Tensor
    scalar_z: torch.Tensor
    vector_z: torch.Tensor
    scalar_mu: torch.Tensor
    vector_mu: torch.Tensor
    scalar_logvar: torch.Tensor
    vector_logvar: torch.Tensor


@dataclass
class T5FactorVAEModelOutput:
    t5_encoder_sequence: torch.Tensor
    attention_weights: torch.Tensor
    pooled_scalar_tensor: torch.Tensor
    vae: FactorVAEOutput
    vae_decoded_sequence: torch.Tensor
    t5_decoder_sequence: torch.Tensor




In [15]:
class T5EncoderBackbone(nn.Module):
    """Loads a pretrained T5 encoder and returns the sequence of hidden states."""

    def __init__(
        self,
        model_name: str = "google/flan-t5-base",
        use_lora: bool = False,
        lora_r: int = 16,
        lora_alpha: int = 32,
        lora_dropout: float = 0.0,
        lora_target_modules: Sequence[str] = ("q", "v"),
    ) -> None:
        super().__init__()
        model = T5EncoderModel.from_pretrained(model_name)
        if use_lora:
            model = get_peft_model(
                model,
                LoraConfig(
                    task_type=TaskType.FEATURE_EXTRACTION,
                    r=lora_r,
                    lora_alpha=lora_alpha,
                    lora_dropout=lora_dropout,
                    target_modules=list(lora_target_modules),
                    bias="none",
                ),
            )
        self.model = model
        self.hidden_size = model.config.d_model

    def forward(
        self,
        input_ids: torch.LongTensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> T5EncoderOutput:
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )
        return T5EncoderOutput(last_hidden_state=outputs.last_hidden_state)




In [16]:
class T5DecoderBackbone(nn.Module):
    """
    Loads a pretrained T5 decoder path and returns the decoder hidden-state sequence.

    The decoder consumes the encoder sequence through cross-attention.
    """

    def __init__(
        self,
        model_name: str = "google/flan-t5-base",
        use_lora: bool = False,
        lora_r: int = 16,
        lora_alpha: int = 32,
        lora_dropout: float = 0.0,
        lora_target_modules: Sequence[str] = ("q", "v"),
    ) -> None:
        super().__init__()
        model = T5ForConditionalGeneration.from_pretrained(model_name)
        if use_lora:
            model = get_peft_model(
                model,
                LoraConfig(
                    task_type=TaskType.SEQ_2_SEQ_LM,
                    r=lora_r,
                    lora_alpha=lora_alpha,
                    lora_dropout=lora_dropout,
                    target_modules=list(lora_target_modules),
                    bias="none",
                ),
            )
        self.model = model
        self.hidden_size = model.config.d_model

    def _decoder_start_tokens(
        self,
        batch_size: int,
        device: torch.device,
    ) -> torch.LongTensor:
        start_token_id = self.model.config.decoder_start_token_id
        if start_token_id is None:
            start_token_id = self.model.config.pad_token_id
        if start_token_id is None:
            raise ValueError(
                "T5 requires either decoder_start_token_id or pad_token_id to create decoder inputs."
            )
        return torch.full(
            (batch_size, 1),
            fill_value=start_token_id,
            dtype=torch.long,
            device=device,
        )

    def forward(
        self,
        encoder_hidden_states: torch.Tensor,
        encoder_attention_mask: Optional[torch.Tensor] = None,
        decoder_input_ids: Optional[torch.LongTensor] = None,
        decoder_attention_mask: Optional[torch.Tensor] = None,
        decoder_inputs_embeds: Optional[torch.Tensor] = None,
    ) -> T5DecoderOutput:
        batch_size = encoder_hidden_states.size(0)
        if decoder_input_ids is None and decoder_inputs_embeds is None:
            decoder_input_ids = self._decoder_start_tokens(
                batch_size=batch_size,
                device=encoder_hidden_states.device,
            )

        outputs = self.model(
            attention_mask=encoder_attention_mask,
            encoder_outputs=BaseModelOutput(last_hidden_state=encoder_hidden_states),
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            decoder_inputs_embeds=decoder_inputs_embeds,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True,
        )
        decoder_states = outputs.decoder_hidden_states
        if decoder_states is None:
            raise RuntimeError("Expected decoder hidden states, but the model returned None.")
        return T5DecoderOutput(
            last_hidden_state=decoder_states[-1],
            all_hidden_states=tuple(decoder_states),
        )




In [23]:
class ScalarLatentAttentionPooling(nn.Module):
    """
    Computes attention weights from a chosen source sequence and pools scalar latents.

    Output pooled scalar tensor shape is always (B, n, H).
    """

    def __init__(
        self,
        num_scalar_factors: int,
        source_dim: int,
        num_heads: int = 4,
        pooling_mode: str = "per_scalar_dim",
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        if pooling_mode not in {"per_scalar_dim", "joint_scalar_vector"}:
            raise ValueError(
                f"Unsupported pooling_mode={pooling_mode}. Expected one of: per_scalar_dim, joint_scalar_vector."
            )
        self.num_scalar_factors = num_scalar_factors
        self.source_dim = source_dim
        self.num_heads = num_heads
        self.pooling_mode = pooling_mode
        self.dropout = nn.Dropout(dropout)

        if pooling_mode == "per_scalar_dim":
            self.score_weight = nn.Parameter(
                torch.empty(num_scalar_factors, num_heads, source_dim)
            )
            self.score_bias = nn.Parameter(torch.zeros(num_scalar_factors, num_heads))
        else:
            self.score_weight = nn.Parameter(torch.empty(num_heads, source_dim))
            self.score_bias = nn.Parameter(torch.zeros(num_heads))
        nn.init.xavier_uniform_(self.score_weight)

    def forward(
        self,
        source_sequence: torch.Tensor,
        scalar_sequence: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        source_sequence = self.dropout(source_sequence)

        if self.pooling_mode == "per_scalar_dim":
            logits = torch.einsum(
                "bsd,nhd->bnhs",
                source_sequence,
                self.score_weight,
            ) + self.score_bias.unsqueeze(0).unsqueeze(-1)
        else:
            shared_logits = torch.einsum(
                "bsd,hd->bhs",
                source_sequence,
                self.score_weight,
            ) + self.score_bias.unsqueeze(0).unsqueeze(-1)
            logits = shared_logits.unsqueeze(1).expand(
                -1,
                self.num_scalar_factors,
                -1,
                -1,
            )

        if attention_mask is not None:
            if attention_mask.dtype != torch.bool:
                keep_mask = attention_mask > 0
            else:
                keep_mask = attention_mask
            logits = logits.masked_fill(
                ~keep_mask.unsqueeze(1).unsqueeze(1),
                torch.finfo(source_sequence.dtype).min,
            )

        weights = torch.softmax(logits, dim=-1)
        values = scalar_sequence.transpose(1, 2).unsqueeze(2)
        pooled = torch.sum(weights * values, dim=-1)
        return pooled, weights




In [18]:
class FactorVAEEncoder(nn.Module):
    """
    Encodes a vector into a split latent space: n scalar 1-D factors + one x-D vector factor.
    """

    def __init__(
        self,
        input_dim: int,
        num_scalar_factors: int,
        vector_latent_dim: int,
        hidden_dim: int = 1024,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.num_scalar_factors = num_scalar_factors
        self.vector_latent_dim = vector_latent_dim
        self.total_latent_dim = num_scalar_factors + vector_latent_dim

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.mu = nn.Linear(hidden_dim, self.total_latent_dim)
        self.logvar = nn.Linear(hidden_dim, self.total_latent_dim)

    def split(self, tensor: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        scalar = tensor[..., : self.num_scalar_factors]
        vector = tensor[..., self.num_scalar_factors :]
        return scalar, vector

    def reparameterize(
        self,
        mu: torch.Tensor,
        logvar: torch.Tensor,
        sample: bool = True,
    ) -> torch.Tensor:
        if not sample:
            return mu
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x: torch.Tensor, sample: bool = True) -> FactorVAEOutput:
        hidden = self.net(x)
        mu = self.mu(hidden)
        logvar = self.logvar(hidden)
        z = self.reparameterize(mu=mu, logvar=logvar, sample=sample)

        scalar_z, vector_z = self.split(z)
        scalar_mu, vector_mu = self.split(mu)
        scalar_logvar, vector_logvar = self.split(logvar)

        return FactorVAEOutput(
            z=z,
            mu=mu,
            logvar=logvar,
            scalar_z=scalar_z,
            vector_z=vector_z,
            scalar_mu=scalar_mu,
            vector_mu=vector_mu,
            scalar_logvar=scalar_logvar,
            vector_logvar=vector_logvar,
        )




In [19]:
class FactorVAEDecoder(nn.Module):
    """Decodes the split latent back into a feature vector."""

    def __init__(
        self,
        output_dim: int,
        num_scalar_factors: int,
        vector_latent_dim: int,
        hidden_dim: int = 1024,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        total_latent_dim = num_scalar_factors + vector_latent_dim
        self.net = nn.Sequential(
            nn.Linear(total_latent_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(
        self,
        scalar_latent: Optional[torch.Tensor] = None,
        vector_latent: Optional[torch.Tensor] = None,
        z: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if z is None:
            if scalar_latent is None or vector_latent is None:
                raise ValueError(
                    "Provide either z or both scalar_latent and vector_latent to the FactorVAEDecoder."
                )
            z = torch.cat([scalar_latent, vector_latent], dim=-1)
        return self.net(z)




In [24]:
class T5FactorVAEModel(nn.Module):
    """
    A merged model with ablations over latent bottleneck, attention source, pooling mode,
    number of attention heads, and encoder skip fusion.
    """

    def __init__(
        self,
        model_name: str = "google/flan-t5-base",
        num_scalar_factors: int = 8,
        vector_latent_dim: int = 64,
        vae_hidden_dim: int = 1024,
        use_lora: bool = False,
        lora_r: int = 16,
        lora_alpha: int = 32,
        lora_dropout: float = 0.0,
        lora_target_modules: Sequence[str] = ("q", "v"),
        pooling_dropout: float = 0.0,
        latent_pool_heads: int = 4,
        attention_source: str = "latent_full",
        pooling_mode: str = "per_scalar_dim",
        use_skip_connection: bool = True,
        vae_dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.num_scalar_factors = num_scalar_factors
        self.vector_latent_dim = vector_latent_dim
        self.attention_source = attention_source
        self.pooling_mode = pooling_mode
        self.use_skip_connection = use_skip_connection

        valid_sources = {"scalar_only", "vector_only", "latent_full", "encoder_sequence"}
        if attention_source not in valid_sources:
            raise ValueError(
                f"Unsupported attention_source={attention_source}. Expected one of: scalar_only, vector_only, latent_full, encoder_sequence."
            )

        self.t5_encoder = T5EncoderBackbone(
            model_name=model_name,
            use_lora=use_lora,
            lora_r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            lora_target_modules=lora_target_modules,
        )
        self.t5_decoder = T5DecoderBackbone(
            model_name=model_name,
            use_lora=use_lora,
            lora_r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            lora_target_modules=lora_target_modules,
        )

        hidden_size = self.t5_encoder.hidden_size
        if hidden_size != self.t5_decoder.hidden_size:
            raise ValueError(
                f"Encoder hidden size {hidden_size} does not match decoder hidden size {self.t5_decoder.hidden_size}."
            )

        self.vae_encoder = FactorVAEEncoder(
            input_dim=hidden_size,
            num_scalar_factors=num_scalar_factors,
            vector_latent_dim=vector_latent_dim,
            hidden_dim=vae_hidden_dim,
            dropout=vae_dropout,
        )

        source_dim_map = {
            "scalar_only": num_scalar_factors,
            "vector_only": vector_latent_dim,
            "latent_full": num_scalar_factors + vector_latent_dim,
            "encoder_sequence": hidden_size,
        }
        self.scalar_attention_pool = ScalarLatentAttentionPooling(
            num_scalar_factors=num_scalar_factors,
            source_dim=source_dim_map[attention_source],
            num_heads=latent_pool_heads,
            pooling_mode=pooling_mode,
            dropout=pooling_dropout,
        )

        self.vae_decoder = FactorVAEDecoder(
            output_dim=hidden_size,
            num_scalar_factors=num_scalar_factors,
            vector_latent_dim=vector_latent_dim,
            hidden_dim=vae_hidden_dim,
            dropout=vae_dropout,
        )
        self.memory_norm = nn.LayerNorm(hidden_size)
        self.memory_gate = nn.Linear(hidden_size, hidden_size)

    def _get_attention_source_sequence(
        self,
        vae_out: FactorVAEOutput,
        t5_encoder_sequence: torch.Tensor,
    ) -> torch.Tensor:
        if self.attention_source == "scalar_only":
            return vae_out.scalar_z
        if self.attention_source == "vector_only":
            return vae_out.vector_z
        if self.attention_source == "latent_full":
            return vae_out.z
        return t5_encoder_sequence

    def fuse_memory(
        self,
        t5_encoder_sequence: torch.Tensor,
        decoded_full: torch.Tensor,
    ) -> torch.Tensor:
        gate = torch.sigmoid(self.memory_gate(decoded_full))
        fused = t5_encoder_sequence + gate * decoded_full
        return self.memory_norm(fused)

    def encode(
        self,
        input_ids: torch.LongTensor,
        attention_mask: Optional[torch.Tensor] = None,
        sample_posterior: bool = True,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, FactorVAEOutput]:
        t5_encoder_sequence = self.t5_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state
        vae_out = self.vae_encoder(t5_encoder_sequence, sample=sample_posterior)
        source_sequence = self._get_attention_source_sequence(
            vae_out=vae_out,
            t5_encoder_sequence=t5_encoder_sequence,
        )
        pooled_scalar_tensor, attn_weights = self.scalar_attention_pool(
            source_sequence=source_sequence,
            scalar_sequence=vae_out.scalar_z,
            attention_mask=attention_mask,
        )
        return t5_encoder_sequence, attn_weights, pooled_scalar_tensor, vae_out

    def decode(
        self,
        vae_out: FactorVAEOutput,
        t5_encoder_sequence: torch.Tensor,
        encoder_attention_mask: Optional[torch.Tensor] = None,
        decoder_input_ids: Optional[torch.LongTensor] = None,
        decoder_attention_mask: Optional[torch.Tensor] = None,
        decoder_inputs_embeds: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        vae_decoded_sequence = self.vae_decoder(z=vae_out.z)
        if self.use_skip_connection:
            decoder_memory = self.fuse_memory(t5_encoder_sequence, vae_decoded_sequence)
        else:
            decoder_memory = vae_decoded_sequence

        decoded = self.t5_decoder(
            encoder_hidden_states=decoder_memory,
            encoder_attention_mask=encoder_attention_mask,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            decoder_inputs_embeds=decoder_inputs_embeds,
        )
        return decoded.last_hidden_state, vae_decoded_sequence

    def forward(
        self,
        input_ids: torch.LongTensor,
        attention_mask: Optional[torch.Tensor] = None,
        decoder_input_ids: Optional[torch.LongTensor] = None,
        decoder_attention_mask: Optional[torch.Tensor] = None,
        decoder_inputs_embeds: Optional[torch.Tensor] = None,
        sample_posterior: bool = True,
    ) -> T5FactorVAEModelOutput:
        (
            t5_encoder_sequence,
            attn_weights,
            pooled_scalar_tensor,
            vae_out,
        ) = self.encode(
            input_ids=input_ids,
            attention_mask=attention_mask,
            sample_posterior=sample_posterior,
        )
        t5_decoder_sequence, vae_decoded_sequence = self.decode(
            vae_out=vae_out,
            t5_encoder_sequence=t5_encoder_sequence,
            encoder_attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            decoder_inputs_embeds=decoder_inputs_embeds,
        )
        return T5FactorVAEModelOutput(
            t5_encoder_sequence=t5_encoder_sequence,
            attention_weights=attn_weights,
            pooled_scalar_tensor=pooled_scalar_tensor,
            vae=vae_out,
            vae_decoded_sequence=vae_decoded_sequence,
            t5_decoder_sequence=t5_decoder_sequence,
        )



In [25]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = T5FactorVAEModel(
        model_name="google/flan-t5-base",
        num_scalar_factors=8,
        vector_latent_dim=64,
        latent_pool_heads=4,
        attention_source="latent_full",
        pooling_mode="per_scalar_dim",
        use_skip_connection=True,
        use_lora=False,
    ).to(device)

    batch_size = 2
    src_len = 12
    tgt_len = 6
    vocab_size = model.t5_encoder.model.config.vocab_size

    input_ids = torch.randint(0, vocab_size, (batch_size, src_len), device=device)
    attention_mask = torch.ones(batch_size, src_len, device=device)
    decoder_input_ids = torch.randint(0, vocab_size, (batch_size, tgt_len), device=device)

    out = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        decoder_input_ids=decoder_input_ids,
        sample_posterior=True,
    )

    print("t5_encoder_sequence:", tuple(out.t5_encoder_sequence.shape))
    print("vae.z (per token):", tuple(out.vae.z.shape))
    print("attention_weights:", tuple(out.attention_weights.shape))
    print("pooled_scalar_tensor (classification features):", tuple(out.pooled_scalar_tensor.shape))
    print("scalar_z:", tuple(out.vae.scalar_z.shape))
    print("vector_z:", tuple(out.vae.vector_z.shape))
    print("vae_decoded_sequence:", tuple(out.vae_decoded_sequence.shape))
    print("t5_decoder_sequence:", tuple(out.t5_decoder_sequence.shape))

Loading weights: 100%|██████████| 111/111 [00:00<00:00, 956.10it/s]
T5EncoderModel LOAD REPORT from: google/flan-t5-base
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 829.52it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


t5_encoder_sequence: (2, 12, 768)
vae.z (per token): (2, 12, 72)
attention_weights: (2, 8, 4, 12)
pooled_scalar_tensor (classification features): (2, 8, 4)
scalar_z: (2, 12, 8)
vector_z: (2, 12, 64)
vae_decoded_sequence: (2, 12, 768)
t5_decoder_sequence: (2, 6, 768)


In [26]:
# Ablation smoke checks: pooling sources/modes and skip-connection toggle
with torch.no_grad():
    b, s = 2, 12
    n = model.num_scalar_factors
    x = model.vector_latent_dim
    h = model.scalar_attention_pool.num_heads
    enc_dim = model.t5_encoder.hidden_size

    scalar_seq = torch.randn(b, s, n, device=device)
    vector_seq = torch.randn(b, s, x, device=device)
    latent_seq = torch.cat([scalar_seq, vector_seq], dim=-1)
    encoder_seq = torch.randn(b, s, enc_dim, device=device)
    mask = torch.ones(b, s, device=device)

    source_map = {
        "scalar_only": scalar_seq,
        "vector_only": vector_seq,
        "latent_full": latent_seq,
        "encoder_sequence": encoder_seq,
    }

    for source_name, source_seq in source_map.items():
        for pooling_mode in ("per_scalar_dim", "joint_scalar_vector"):
            pool = ScalarLatentAttentionPooling(
                num_scalar_factors=n,
                source_dim=source_seq.size(-1),
                num_heads=h,
                pooling_mode=pooling_mode,
            ).to(device)
            pooled, weights = pool(
                source_sequence=source_seq,
                scalar_sequence=scalar_seq,
                attention_mask=mask,
            )
            assert pooled.shape == (b, n, h), (source_name, pooling_mode, pooled.shape)
            assert weights.shape == (b, n, h, s), (source_name, pooling_mode, weights.shape)

    model.use_skip_connection = False
    out_no_skip = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        decoder_input_ids=decoder_input_ids,
        sample_posterior=True,
    )
    assert out_no_skip.t5_decoder_sequence.shape[:2] == (batch_size, tgt_len)
    model.use_skip_connection = True

print("Ablation smoke checks passed for all sources/modes and skip toggle.")

Ablation smoke checks passed for all sources/modes and skip toggle.
